# Caso Práctico — Mercado Libre
## Entregable 1: Consolidación de Datos con Python

### Objetivo

Construir un flujo de carga y transformación de datos que permita consolidar la información de los Service Centers, generar indicadores relevantes y preparar una base limpia que posteriormente pueda ser consumida por un dashboard.

### Flujo del proceso

1. Carga de Información
2. Revisión Inicial de los datos 
3. Limpieza y validaciones
4. Transformación y cálculo de métricas
5. Construcción de dataset final
6. Exportación para consumo de dashboard

In [2]:
# ==========================================================
# CONFIGURACIÓN DE RUTAS DEL PROYECTO
# ==========================================================

import os

# Ruta base del proyecto (donde está el notebook)
ruta_base = os.getcwd()

# Crear carpetas automáticamente si no existen
carpeta_datos = os.path.join(ruta_base,"data")
carpeta_salida = os.path.join(ruta_base,"output")

os.makedirs(carpeta_datos, exist_ok=True)
os.makedirs(carpeta_salida, exist_ok=True)

print("Carpetas verificadas correctamente")

Carpetas verificadas correctamente


## 1. Importación de librerías

Se cargan las librerías necesarias para manipulación de datos y manejo de fechas que serán utilizadas durante el proceso de transformación.

In [3]:
# Librerías para manipulación y análisis de datos
import pandas as pd
import numpy as np

# Manejo de fechas
from datetime import datetime

# Visualización de las columnas
pd.set_option('display.max_columns',None)

## 2. Carga de datos

Se carga el archivo proporcionado y se realiza una revisión inicial para confirmar que la estructura fue leída correctamente.

In [4]:
archivo = "service_centers_dataset.csv"

ruta_archivo = os.path.join(
    carpeta_datos,
    archivo
)

# Valida existencia del archivo
if os.path.exists(ruta_archivo):
    try:

        df = pd.read_csv(ruta_archivo)

        print("Archivo cargado correctamente")

    except Exception as e:

        print(
            f"Error durante la carga: {e}"
        )

else:

    print(
        f"No se encontró el archivo: {archivo}"
    )

    print(
        f"Coloca el archivo dentro de:\n{carpeta_datos}"
    )

display(df.head())

Archivo cargado correctamente


,id_sc,nombre_sc,estado,tipo_sc,m2_operativos,throughput_diario_meta,throughput_real_ult_semana,avance_obra_civil_pct,avance_compras_pct,avance_licencias_pct,fecha_estimada_apertura,fecha_apertura_real,dias_retraso,nivel_automatizacion,mix_gm_pct,mix_large_pct,mix_oversize_pct,capex_presupuestado_usd,capex_ejercido_usd,estatus_general,observaciones
0,SC-001,SMX-Norte,CDMX,Mediano,1800,3500,3200,100,95,80,2026-02-15,NaN,0,semi,72,20,8,850000,780000,En operación piloto,Licencias pendientes de validación final
1,SC-002,SMX-Sur,CDMX,Grande,3200,6000,0,85,60,30,2026-04-10,NaN,35,semi,70,20,10,1250000,620000,En habilitación,Retraso en entrega de equipamiento por proveedor
2,SC-003,GDL-Zapopan,Jalisco,Mediano,2100,4000,0,100,90,95,2026-03-01,NaN,18,semi,75,18,7,920000,890000,Pre-apertura,Sorter con falla en instalación — reemplazo en...
3,SC-004,MTY-San Nicolás,Nuevo León,Grande,4000,8000,0,60,40,20,2026-06-20,NaN,0,full,68,22,10,1800000,540000,En obra civil,Sin retrasos previstos a la fecha
4,SC-005,QRO-Centro,Querétaro,Pequeño,750,1500,0,100,100,100,2026-01-30,2026-02-05,0,manual,78,15,7,380000,395000,Operativo,Apertura completada — sobre presupuesto por aj...


## 3. Revisión inicial de la información

Antes de comenzar con cálculos o transformaciones se realiza una revisión rápida para entender la estructura del dataset e identificar posibles inconsistencias como: 

- Número de registros y columnas
- Tipos de datos
- Valores nulos
- Registros duplicados
- Primeras observaciones de calidad de datos

In [5]:
# Dimensión general del archivo
print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")

# Revisar estructura y tipos de datos
print("\nTipos de datos encontrados:")
display(df.dtypes)

# Revisar valores nulos
print("\nValores faltantes por columna:")
display(df.isnull().sum())

# Revisar registros duplicados
print("\nRegistros duplicados encontrados:")
print(df.duplicated().sum())

Filas: 12
Columnas: 21

Tipos de datos encontrados:


id_sc                         object
nombre_sc                     object
estado                        object
tipo_sc                       object
m2_operativos                  int64
throughput_diario_meta         int64
throughput_real_ult_semana     int64
avance_obra_civil_pct          int64
avance_compras_pct             int64
avance_licencias_pct           int64
fecha_estimada_apertura       object
fecha_apertura_real           object
dias_retraso                   int64
nivel_automatizacion          object
mix_gm_pct                     int64
mix_large_pct                  int64
mix_oversize_pct               int64
capex_presupuestado_usd        int64
capex_ejercido_usd             int64
estatus_general               object
observaciones                 object
dtype: object


Valores faltantes por columna:


id_sc                          0
nombre_sc                      0
estado                         0
tipo_sc                        0
m2_operativos                  0
throughput_diario_meta         0
throughput_real_ult_semana     0
avance_obra_civil_pct          0
avance_compras_pct             0
avance_licencias_pct           0
fecha_estimada_apertura        0
fecha_apertura_real           11
dias_retraso                   0
nivel_automatizacion           0
mix_gm_pct                     0
mix_large_pct                  0
mix_oversize_pct               0
capex_presupuestado_usd        0
capex_ejercido_usd             0
estatus_general                0
observaciones                  0
dtype: int64


Registros duplicados encontrados:
0


## Hallazgos iniciales

Después de revisar el dataset se identificaron algunos puntos importantes:

- El archivo contiene información correspondiente a 12 Service Centers.
- Existen registros sin fecha_apertura_real, lo cual puede ser esperado debido a que algunos centros aún se encuentran en proceso de habilitación.
- No se identificaron duplicados a nivel registro.
- Será necesario convertir columnas de fecha y validar algunos indicadores porcentuales antes de generar métricas consolidadas.

## 4. Limpieza y preparación de datos

Antes de generar indicadores se realizan algunas validaciones y ajustes básicos para trabajar sobre una estructura consistente y evitar errores durante los cálculos posteriores. Los ajustes a realizar son:

- Estandarización de nombres de columnas
- Conversión de fechas
- Limpieza de textos
- Validación de duplicados por ID de Service Center

In [6]:
# Crear una copia para trabajar sobre ella y mantener intacta la fuente original
df_clean = df.copy()

# Estandarizar nombres de columnas
df_clean.columns = (
    df_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(" ","_")
)

# Convertir columnas de fecha para utilizarlas en cálculos posteriores
columnas_fecha = [
    "fecha_estimada_apertura",
    "fecha_apertura_real"
]

for col in columnas_fecha:

    df_clean[col] = pd.to_datetime(
        df_clean[col],
        errors="coerce"
    )

# Validar duplicados por identificador principal
duplicados = df_clean[
    df_clean.duplicated(
        subset=["id_sc"],
        keep=False
    )
]

print(
    f"Service Centers duplicados encontrados: {duplicados.shape[0]}"
)

# Eliminar duplicados si existen
df_clean = df_clean.drop_duplicates(
    subset=["id_sc"]
)

Service Centers duplicados encontrados: 0


## 5. Validación de calidad de datos

Se realizan validaciones para detectar inconsistencias en campos clave:

- Porcentajes fuera del rango 0% a 100%
- Fechas inválidas
- Valores negativos en métricas operativas o financieras
- Mix logístico diferente de 100%

In [7]:
# Columnas de avance porcentual
columnas_pct = [
    "avance_obra_civil_pct",
    "avance_compras_pct",
    "avance_licencias_pct",
    "mix_gm_pct",
    "mix_large_pct",
    "mix_oversize_pct"
]

# Validar porcentajes fuera de rango
validacion_pct = {}

for col in columnas_pct:
    fuera_rango = df_clean[(df_clean[col] < 0) | (df_clean[col] > 100)]
    validacion_pct[col] = fuera_rango.shape[0]

print("Valores porcentuales fuera de rango:")
display(pd.Series(validacion_pct))

# Validar fechas estimadas nulas
print("\nFechas estimadas de apertura nulas:")
print(df_clean["fecha_estimada_apertura"].isnull().sum())

# Validar valores negativos en campos numéricos clave
columnas_numericas_clave = [
    "m2_operativos",
    "throughput_diario_meta",
    "throughput_real_ult_semana",
    "capex_presupuestado_usd",
    "capex_ejercido_usd"
]

validacion_negativos = {}

for col in columnas_numericas_clave:
    negativos = df_clean[df_clean[col] < 0]
    validacion_negativos[col] = negativos.shape[0]

print("\nValores negativos en métricas clave:")
display(pd.Series(validacion_negativos))

# Validar que el mix sume 100%
df_clean["mix_total_pct"] = (
    df_clean["mix_gm_pct"] +
    df_clean["mix_large_pct"] +
    df_clean["mix_oversize_pct"]
)

print("\nRegistros donde el mix no suma 100%:")
display(df_clean[df_clean["mix_total_pct"] != 100][["id_sc", "mix_total_pct"]])

Valores porcentuales fuera de rango:


avance_obra_civil_pct    0
avance_compras_pct       0
avance_licencias_pct     0
mix_gm_pct               0
mix_large_pct            0
mix_oversize_pct         0
dtype: int64


Fechas estimadas de apertura nulas:
0

Valores negativos en métricas clave:


m2_operativos                 0
throughput_diario_meta        0
throughput_real_ult_semana    0
capex_presupuestado_usd       0
capex_ejercido_usd            0
dtype: int64


Registros donde el mix no suma 100%:


,id_sc,mix_total_pct


## Tratamiento de inconsistencias detectadas

Con base en las validaciones anteriores, se aplican reglas de limpieza para corregir valores fuera de rango, negativos o inconsistentes antes de construir el dataset final.

In [8]:
# Crear copia para tratamiento
df_clean = df_clean.copy()

# 1. Limpia porcentajes fuera de rango
# Los porcentajes deben estar entre 0 y 100

for col in columnas_pct:
    df_clean[col] = df_clean[col].clip(lower=0, upper=100)


# 2. Limpia valores negativos en métricas numéricas clave

for col in columnas_numericas_clave:
    df_clean[col] = df_clean[col].clip(lower=0)


# 3. Validar fechas nulas
# En este caso no se imputan fechas estimadas porque podrían alterar la lógica del negocio.

df_clean["fecha_estimada_apertura_nula"] = df_clean["fecha_estimada_apertura"].isna()


# 4. Recalcular mix total después de limpiar porcentajes

df_clean["mix_total_pct"] = (
    df_clean["mix_gm_pct"] +
    df_clean["mix_large_pct"] +
    df_clean["mix_oversize_pct"]
)

## 6. Generación de métricas

En esta sección se generan los indicadores principales del caso, tomando como base
- % de Avances por categoría
- Fechas estimadas de apertura
- Días de retraso acumulado

In [9]:
# Columnas de avance solicitadas en el caso
columnas_avance = [
    "avance_obra_civil_pct",
    "avance_compras_pct",
    "avance_licencias_pct"
]

# Calcular avance total promedio por Service Center
df_clean["avance_total_pct"] = df_clean[columnas_avance].mean(axis=1).round(2)

# Recalcular días de retraso tomando como referencia la fecha actual
fecha_actual = pd.to_datetime(datetime.today().date())

df_clean["dias_retraso_calculado"] = (
    fecha_actual - df_clean["fecha_estimada_apertura"]
).dt.days

# Si la fecha aún no está vencida, el retraso se considera 0
df_clean["dias_retraso_calculado"] = df_clean[
    "dias_retraso_calculado"
].clip(lower=0)

# Clasificación de retraso para lectura en dashboard
df_clean["estatus_retraso"] = np.where(
    df_clean["dias_retraso_calculado"] > 0,
    "Con retraso",
    "En tiempo"
)

# Clasificación de avance para lectura en dashboard
df_clean["nivel_avance"] = pd.cut(
    df_clean["avance_total_pct"],
    bins=[0, 50, 80, 100],
    labels=["Bajo", "Medio", "Alto"],
    include_lowest=True
)

display(df_clean.head())

,id_sc,nombre_sc,estado,tipo_sc,m2_operativos,throughput_diario_meta,throughput_real_ult_semana,avance_obra_civil_pct,avance_compras_pct,avance_licencias_pct,fecha_estimada_apertura,fecha_apertura_real,dias_retraso,nivel_automatizacion,mix_gm_pct,mix_large_pct,mix_oversize_pct,capex_presupuestado_usd,capex_ejercido_usd,estatus_general,observaciones,mix_total_pct,fecha_estimada_apertura_nula,avance_total_pct,dias_retraso_calculado,estatus_retraso,nivel_avance
0,SC-001,SMX-Norte,CDMX,Mediano,1800,3500,3200,100,95,80,2026-02-15,NaT,0,semi,72,20,8,850000,780000,En operación piloto,Licencias pendientes de validación final,100,False,91.67,100,Con retraso,Alto
1,SC-002,SMX-Sur,CDMX,Grande,3200,6000,0,85,60,30,2026-04-10,NaT,35,semi,70,20,10,1250000,620000,En habilitación,Retraso en entrega de equipamiento por proveedor,100,False,58.33,46,Con retraso,Medio
2,SC-003,GDL-Zapopan,Jalisco,Mediano,2100,4000,0,100,90,95,2026-03-01,NaT,18,semi,75,18,7,920000,890000,Pre-apertura,Sorter con falla en instalación — reemplazo en...,100,False,95.00,86,Con retraso,Alto
3,SC-004,MTY-San Nicolás,Nuevo León,Grande,4000,8000,0,60,40,20,2026-06-20,NaT,0,full,68,22,10,1800000,540000,En obra civil,Sin retrasos previstos a la fecha,100,False,40.00,0,En tiempo,Bajo
4,SC-005,QRO-Centro,Querétaro,Pequeño,750,1500,0,100,100,100,2026-01-30,2026-02-05,0,manual,78,15,7,380000,395000,Operativo,Apertura completada — sobre presupuesto por aj...,100,False,100.00,116,Con retraso,Alto


## 7. Métricas adicionales para dashboard

Además de los campos mínimos solicitados, se agregan indicadores operativos y financieros que pueden ayudar a priorizar la revisión semanal del equipo.

In [10]:
# Cumplimiento de throughput contra la meta diaria
df_clean["cumplimiento_throughput_pct"] = np.where(
    df_clean["throughput_diario_meta"] > 0,
    (
        df_clean["throughput_real_ult_semana"] /
        df_clean["throughput_diario_meta"]
    ) * 100,
    0
).round(2)

# Avance de ejecución CAPEX
df_clean["ejecucion_capex_pct"] = np.where(
    df_clean["capex_presupuestado_usd"] > 0,
    (
        df_clean["capex_ejercido_usd"] /
        df_clean["capex_presupuestado_usd"]
    ) * 100,
    0
).round(2)

# Clasificación de riesgo de apertura considerando retraso y Service Center  operativos
df_clean["riesgo_apertura"] = np.select(
    [
        # SCs que ya tienen fecha real de apertura
        df_clean["fecha_apertura_real"].notna(),

        # Riesgo alto: retraso crítico mayor a 14 días
        df_clean["dias_retraso_calculado"] > 14,

        # Riesgo medio: retraso moderado entre 7 y 14 días
        df_clean["dias_retraso_calculado"].between(7, 14),

        # Riesgo bajo: sin retraso
        df_clean["dias_retraso_calculado"] == 0
    ],
    [
        "Operativo",
        "Alto",
        "Medio",
        "Bajo"
    ],
    default="Bajo"
)

display(
    df_clean[
        [
            "id_sc",
            "nombre_sc",
            "estado",
            "avance_total_pct",
            "dias_retraso_calculado",
            "estatus_retraso",
            "nivel_avance",
            "riesgo_apertura",
            "cumplimiento_throughput_pct",
            "ejecucion_capex_pct"
        ]
    ]
)

,id_sc,nombre_sc,estado,avance_total_pct,dias_retraso_calculado,estatus_retraso,nivel_avance,riesgo_apertura,cumplimiento_throughput_pct,ejecucion_capex_pct
0,SC-001,SMX-Norte,CDMX,91.67,100,Con retraso,Alto,Alto,91.43,91.76
1,SC-002,SMX-Sur,CDMX,58.33,46,Con retraso,Medio,Alto,0.00,49.60
2,SC-003,GDL-Zapopan,Jalisco,95.00,86,Con retraso,Alto,Alto,0.00,96.74
3,SC-004,MTY-San Nicolás,Nuevo León,40.00,0,En tiempo,Bajo,Bajo,0.00,30.00
4,SC-005,QRO-Centro,Querétaro,100.00,116,Con retraso,Alto,Operativo,0.00,103.95
5,SC-006,PUE-Angelópolis,Puebla,56.67,21,Con retraso,Medio,Alto,0.00,53.93
6,SC-007,SLP-Industrial,San Luis Potosí,83.33,72,Con retraso,Alto,Alto,0.00,85.71
7,SC-008,TIJ-Otay,Baja California,28.33,0,En tiempo,Bajo,Bajo,0.00,18.79
8,SC-009,MER-Centro,CDMX,75.00,36,Con retraso,Medio,Alto,0.00,74.67
9,SC-010,VER-Boca del Río,Veracruz,95.00,87,Con retraso,Alto,Alto,0.00,95.16


## 8. Construcción del DataFrame consolidado

Se genera una tabla final con los campos solicitados para el entregable, más algunos indicadores adicionales que pueden aportar valor al tablero.

In [11]:
df_consolidado = df_clean[
    [
        "id_sc",
        "nombre_sc",
        "estado",
        "tipo_sc",
        "avance_obra_civil_pct",
        "avance_compras_pct",
        "avance_licencias_pct",
        "avance_total_pct",
        "fecha_estimada_apertura",
        "fecha_apertura_real",
        "dias_retraso_calculado",
        "estatus_retraso",
        "nivel_avance",
        "riesgo_apertura",
        "cumplimiento_throughput_pct",
        "ejecucion_capex_pct",
        "estatus_general",
        "observaciones"
    ]
].copy()

# Renombrar columna calculada para dejarla con nombre final
df_consolidado = df_consolidado.rename(
    columns={
        "dias_retraso_calculado": "dias_retraso"
    }
)

display(df_consolidado)

,id_sc,nombre_sc,estado,tipo_sc,avance_obra_civil_pct,avance_compras_pct,avance_licencias_pct,avance_total_pct,fecha_estimada_apertura,fecha_apertura_real,dias_retraso,estatus_retraso,nivel_avance,riesgo_apertura,cumplimiento_throughput_pct,ejecucion_capex_pct,estatus_general,observaciones
0,SC-001,SMX-Norte,CDMX,Mediano,100,95,80,91.67,2026-02-15,NaT,100,Con retraso,Alto,Alto,91.43,91.76,En operación piloto,Licencias pendientes de validación final
1,SC-002,SMX-Sur,CDMX,Grande,85,60,30,58.33,2026-04-10,NaT,46,Con retraso,Medio,Alto,0.00,49.60,En habilitación,Retraso en entrega de equipamiento por proveedor
2,SC-003,GDL-Zapopan,Jalisco,Mediano,100,90,95,95.00,2026-03-01,NaT,86,Con retraso,Alto,Alto,0.00,96.74,Pre-apertura,Sorter con falla en instalación — reemplazo en...
3,SC-004,MTY-San Nicolás,Nuevo León,Grande,60,40,20,40.00,2026-06-20,NaT,0,En tiempo,Bajo,Bajo,0.00,30.00,En obra civil,Sin retrasos previstos a la fecha
4,SC-005,QRO-Centro,Querétaro,Pequeño,100,100,100,100.00,2026-01-30,2026-02-05,116,Con retraso,Alto,Operativo,0.00,103.95,Operativo,Apertura completada — sobre presupuesto por aj...
5,SC-006,PUE-Angelópolis,Puebla,Mediano,70,55,45,56.67,2026-05-05,NaT,21,Con retraso,Medio,Alto,0.00,53.93,En habilitación,Demora en permisos municipales — riesgo de +30...
6,SC-007,SLP-Industrial,San Luis Potosí,Pequeño,95,85,70,83.33,2026-03-15,NaT,72,Con retraso,Alto,Alto,0.00,85.71,Pre-apertura,Proveedor de mesas de clasificación entrega tarde
7,SC-008,TIJ-Otay,Baja California,Grande,45,30,10,28.33,2026-08-01,NaT,0,En tiempo,Bajo,Bajo,0.00,18.79,En obra civil,Proyecto en tiempo — riesgo por temporada de l...
8,SC-009,MER-Centro,CDMX,Especializado,90,75,60,75.00,2026-04-20,NaT,36,Con retraso,Medio,Alto,0.00,74.67,En habilitación,SC Large+OS — requiere zona de voluminosos adi...
9,SC-010,VER-Boca del Río,Veracruz,Pequeño,100,100,85,95.00,2026-02-28,NaT,87,Con retraso,Alto,Alto,0.00,95.16,Pre-apertura,Licencia de funcionamiento en trámite final


## 9. Exportación del resultado

El resultado consolidado se exporta para que pueda ser consumido posteriormente por una herramienta de visualización.

Para este MVP se consideran dos salidas:

- Archivo local en Excel y CSV como respaldo reproducible.

In [12]:
# Definir rutas de salida
ruta_excel = os.path.join(
    carpeta_salida,
    "service_centers_consolidado.xlsx"
)

ruta_csv = os.path.join(
    carpeta_salida,
    "service_centers_consolidado.csv"
)

# Exportar resultado a Excel
df_consolidado.to_excel(
    ruta_excel,
    index=False
)

# Exportar resultado a CSV
df_consolidado.to_csv(
    ruta_csv,
    index=False,
    encoding="utf-8-sig"
)

print("Archivos exportados correctamente")

print("\nArchivo Excel:")
print(ruta_excel)

print("\nArchivo CSV:")
print(ruta_csv)

Archivos exportados correctamente

Archivo Excel:
C:\Users\eder.aguirre\Documents\MELI - Service Centers\output\service_centers_consolidado.xlsx

Archivo CSV:
C:\Users\eder.aguirre\Documents\MELI - Service Centers\output\service_centers_consolidado.csv


In [13]:
# ==========================================================
# RESUMEN FINAL DEL PROCESAMIENTO
# ==========================================================

print("="*50)
print("Resumen del procesamiento")
print("="*50)

print(f"Service Centers procesados: {df_consolidado.shape[0]}")

print(
    f"Promedio avance total: "
    f"{round(df_consolidado['avance_total_pct'].mean(),2)}%"
)

print(
    f"Centros con retraso: "
    f"{(df_consolidado['estatus_retraso']=='Con retraso').sum()}"
)

print(
    f"Centros de riesgo alto: "
    f"{(df_consolidado['riesgo_apertura']=='Alto').sum()}"
)

print("="*50)

Resumen del procesamiento
Service Centers procesados: 12
Promedio avance total: 65.28%
Centros con retraso: 8
Centros de riesgo alto: 7


# Entregable 3 - Módulo de IA para detección de riesgos

En esta sección se construye un módulo de inteligencia artificial que analiza la información operativa consolidada de los Service Centers.

El objetivo es generar automáticamente:
- Un resumen ejecutivo semanal.
- Los 3 principales riesgos operativos.
- Acciones concretas para los Service Centers con mayor retraso.

In [14]:
# ===========================================
# Preparación de datos para módulo IA

# Se copia dataframe consolidado a uno alterno para no modificar el original

df_ia = df_consolidado.copy()


# Excluir Service Centers ya operativos
# porque ya no representan riesgo de apertura

df_ia = df_ia[
    df_ia["estatus_general"] != "Operativo"
]


# Selección de columnas relevantes

df_ia = df_ia[
    [
        "nombre_sc",
        "estado",
        "dias_retraso",
        "riesgo_apertura",
        "cumplimiento_throughput_pct",
        "ejecucion_capex_pct",
        "avance_total_pct",
        "estatus_general",
        "observaciones"
    ]
]


# Ordenamiento por prioridad

df_ia = df_ia.sort_values(
    by=[
        "dias_retraso",
        "cumplimiento_throughput_pct"
    ],
    ascending=[False, True]
)

display(df_ia.head())

,nombre_sc,estado,dias_retraso,riesgo_apertura,cumplimiento_throughput_pct,ejecucion_capex_pct,avance_total_pct,estatus_general,observaciones
0,SMX-Norte,CDMX,100,Alto,91.43,91.76,91.67,En operación piloto,Licencias pendientes de validación final
9,VER-Boca del Río,Veracruz,87,Alto,0.00,95.16,95.00,Pre-apertura,Licencia de funcionamiento en trámite final
2,GDL-Zapopan,Jalisco,86,Alto,0.00,96.74,95.00,Pre-apertura,Sorter con falla en instalación — reemplazo en...
6,SLP-Industrial,San Luis Potosí,72,Alto,0.00,85.71,83.33,Pre-apertura,Proveedor de mesas de clasificación entrega tarde
1,SMX-Sur,CDMX,46,Alto,0.00,49.60,58.33,En habilitación,Retraso en entrega de equipamiento por proveedor


In [15]:
# Conversión del DataFrame a JSON

df_json = df_ia.to_json(
    orient="records",
    force_ascii=False
)

print(df_json[:1000])

[{"nombre_sc":"SMX-Norte","estado":"CDMX","dias_retraso":100,"riesgo_apertura":"Alto","cumplimiento_throughput_pct":91.43,"ejecucion_capex_pct":91.76,"avance_total_pct":91.67,"estatus_general":"En operación piloto","observaciones":"Licencias pendientes de validación final"},{"nombre_sc":"VER-Boca del Río","estado":"Veracruz","dias_retraso":87,"riesgo_apertura":"Alto","cumplimiento_throughput_pct":0.0,"ejecucion_capex_pct":95.16,"avance_total_pct":95.0,"estatus_general":"Pre-apertura","observaciones":"Licencia de funcionamiento en trámite final"},{"nombre_sc":"GDL-Zapopan","estado":"Jalisco","dias_retraso":86,"riesgo_apertura":"Alto","cumplimiento_throughput_pct":0.0,"ejecucion_capex_pct":96.74,"avance_total_pct":95.0,"estatus_general":"Pre-apertura","observaciones":"Sorter con falla en instalación — reemplazo en camino"},{"nombre_sc":"SLP-Industrial","estado":"San Luis Potosí","dias_retraso":72,"riesgo_apertura":"Alto","cumplimiento_throughput_pct":0.0,"ejecucion_capex_pct":85.71,"avan

### Nota de ejecución

El módulo utiliza la API de OpenAI cuando existe una API Key configurada mediante variables de entorno.

En caso de no contar con credenciales, se ejecuta automáticamente un modo demostración para permitir la reproducción completa del notebook sin configuraciones adicionales.

In [16]:
from dotenv import load_dotenv
from openai import OpenAI
import os


# Carga variables del archivo .env
load_dotenv()


# Obtiene API Key
api_key = os.getenv("OPENAI_API_KEY")


In [17]:
# ============================================
# Función IA para detección de riesgos

def generar_resumen_riesgos(df_json):
    # Si existe API usa OpenAI
    if api_key:
        client = OpenAI(api_key=api_key)

        prompt = f"""
        Eres un analista experto en operaciones logísticas de Mercado Libre

        Analiza el siguiente reporte semanal de Service Centers.


        Genera:

        1. Un resumen ejecutivo (máximo 80 palabras).
        2. Los 3 principales riesgos operativos detectados: 
        Para cada riesgo:
        - Service Center afectado
        - Impacto operativo
        - Acción recomendada
        3. Prioriza los Service Centers con:
        - mayor retraso
        - menor avance total
        - riesgo de apertura
        - menor cumplimiento de throughput
        - menor ejecución CAPEX


        Responde en español usando el siguiente formato:

        Resumen Ejecutivo:
        ...

        Riesgo 1:
        ...
        Acción:
        ...

        Riesgo 2:
        ...
        Acción:
        ...

        Riesgo 3:
        ...
        Acción:
        ...

        Datos:
        {df_json}
        """

        respuesta = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {
                    "role":"system",
                    "content":"Eres un experto en logística, BI y análisis operacional."
                },
                {
                    "role":"user",
                    "content":prompt
                }
            ],
            temperature=0.3
        )

        return respuesta.choices[0].message.content
    
    # Si NO existe API, se muestra un ejemplo de como sería el resultado
    else:

        return """
        Resumen Ejecutivo:

        Se identificaron Service Centers con retrasos importantes y bajo desempeño operativo. Los riesgos principales se relacionan con tiempos de apertura, cumplimiento de throughput y ejecución presupuestal.

        Riesgo 1:
        Retrasos críticos de apertura.

        Acción:
        Asignar seguimiento prioritario y recursos adicionales.

        Riesgo 2:
        Bajo cumplimiento de throughput.

        Acción:
        Revisar capacidad operativa y disponibilidad de recursos.

        Riesgo 3:
        Baja ejecución CAPEX.

        Acción:
        Priorizar inversiones críticas.
        """

In [18]:
# Ejecución del análisis IA
from IPython.display import Markdown, display
resultado_ia = generar_resumen_riesgos(df_json)

display(Markdown(resultado_ia))

**Resumen Ejecutivo:**
El reporte semanal de Service Centers muestra un alto riesgo de apertura en varios centros, con retrasos significativos en la implementación. Los Service Centers en CDMX y Veracruz son los más afectados, con un cumplimiento de throughput del 0% en varios casos. Se recomienda priorizar la resolución de licencias y la entrega de equipamiento para mitigar los riesgos operativos.

**Riesgo 1:**
- **Service Center afectado:** VER-Boca del Río
- **Impacto operativo:** Sin cumplimiento de throughput, afectando la capacidad de operación.
- **Acción:** Acelerar la obtención de la licencia de funcionamiento.

**Riesgo 2:**
- **Service Center afectado:** GDL-Zapopan
- **Impacto operativo:** Sin cumplimiento de throughput, retrasando la apertura y operación.
- **Acción:** Priorizar el reemplazo del sorter y asegurar su instalación inmediata.

**Riesgo 3:**
- **Service Center afectado:** SLP-Industrial
- **Impacto operativo:** Sin cumplimiento de throughput, lo que impide la operatividad del centro.
- **Acción:** Coordinar con el proveedor para la entrega urgente de mesas de clasificación.

**Prioridades de Service Centers:**
- **Mayor retraso:** SMX-Norte (100 días)
- **Menor avance total:** AGS-Desarrollo (18.33%)
- **Riesgo de apertura:** SMX-Norte, VER-Boca del Río, GDL-Zapopan (todos con riesgo alto)
- **Menor cumplimiento de throughput:** VER-Boca del Río, GDL-Zapopan, SLP-Industrial (0%)
- **Menor ejecución CAPEX:** TIJ-Otay (18.79%)